# rpg_conv Demo Notebook

This notebook provides reproducible examples for resolving protein marker aliases to canonical gene symbols using `rpg_conv`.

It demonstrates:
- deterministic ground-truth mappings
- robust normalization behavior
- database-backed extension with custom aliases
- optional Ensembl bootstrap flow
- a simple batch-resolution workflow

In [1]:
from pathlib import Path
import sqlite3
import importlib
import importlib.util
import subprocess
import sys


def _find_repo_root(start: Path) -> Path | None:
    """Find repo root containing pyproject.toml and src/rpg_conv."""
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'rpg_conv').exists():
            return candidate
    return None


if importlib.util.find_spec('rpg_conv') is None:
    cwd = Path.cwd().resolve()
    repo_root = _find_repo_root(cwd)

    if repo_root is not None:
        print(f'Installing editable package from: {repo_root}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_root)])
    else:
        # Fallback to published package.
        print('Local source not found; installing from PyPI: rpg_conv')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rpg_conv'])

    importlib.invalidate_caches()

from rpg_conv import GeneResolver
from rpg_conv.normalize import normalize_marker

# Use an isolated local DB to make this notebook reproducible.
DB_PATH = Path('./demo_markers.sqlite3')
if DB_PATH.exists():
    DB_PATH.unlink()

resolver = GeneResolver(db_path=DB_PATH, auto_seed=True)
print(f'Database initialized at: {DB_PATH.resolve()}')

Database initialized at: /Users/k23030440/robust_protein_gene/demo_markers.sqlite3


## 1) Ground-truth checks (deterministic)

These assertions are the core expected behavior.

In [3]:
assert resolver.resolve_one('ki--67') == 'MKI67'
assert resolver.resolve_one('ki67') == 'MKI67'
assert resolver.resolve_one('SMA') == 'ACTA2'
assert resolver.resolve_one('CD57') == 'B3GAT1'

print('All required ground-truth checks passed.')

All required ground-truth checks passed.


## 2) Robust normalization behavior

The resolver normalizes punctuation, spaces, and case before lookup.

In [4]:
variants = ['KI 67', 'ki-67', 'Ki--67', ' ki67 ', 'KI67']
for v in variants:
    print(f"{v!r:10} -> normalized={normalize_marker(v)!r:8} -> gene={resolver.resolve_one(v)}")

'KI 67'    -> normalized='ki67'   -> gene=MKI67
'ki-67'    -> normalized='ki67'   -> gene=MKI67
'Ki--67'   -> normalized='ki67'   -> gene=MKI67
' ki67 '   -> normalized='ki67'   -> gene=MKI67
'KI67'     -> normalized='ki67'   -> gene=MKI67


## 3) Full resolution payload

`resolve()` returns metadata including normalized query, matched alias, and source.

In [5]:
result = resolver.resolve('a-sma')
result

ResolutionResult(query='a-sma', normalized_query='asma', gene_symbol='ACTA2', matched_alias='A-SMA', source='seed')

## 4) Add project-specific aliases

You can extend the DB with your own aliases and keep provenance in `source`.

In [6]:
resolver.add_alias('EPCAM', 'Epithelial Cell Adhesion Molecule', source='custom:demo')
resolver.add_alias('EPCAM', 'ep-cam', source='custom:demo')

print(resolver.resolve('ep-cam'))
print(resolver.resolve('Epithelial Cell Adhesion Molecule'))

ResolutionResult(query='ep-cam', normalized_query='epcam', gene_symbol='EPCAM', matched_alias='ep-cam', source='custom:demo')
ResolutionResult(query='Epithelial Cell Adhesion Molecule', normalized_query='epithelialcelladhesionmolecule', gene_symbol='EPCAM', matched_alias='Epithelial Cell Adhesion Molecule', source='custom:demo')


## 5) Inspect the underlying SQLite database

This confirms the query layer is DB-backed and shows what is stored.

In [7]:
with sqlite3.connect(DB_PATH) as conn:
    total = conn.execute('SELECT COUNT(*) FROM gene_alias').fetchone()[0]
    sample = conn.execute(
        """
        SELECT gene_symbol, alias_raw, alias_norm, source
        FROM gene_alias
        ORDER BY id
        LIMIT 12
        """
    ).fetchall()

print(f'Total rows in gene_alias: {total}')
for row in sample:
    print(row)

Total rows in gene_alias: 11
('MKI67', 'MKI67', 'mki67', 'seed')
('MKI67', 'KI--67', 'ki67', 'seed')
('ACTA2', 'ACTA2', 'acta2', 'seed')
('ACTA2', 'SMA', 'sma', 'seed')
('ACTA2', 'A-SMA', 'asma', 'seed')
('ACTA2', 'ALPHA-SMA', 'alphasma', 'seed')
('B3GAT1', 'B3GAT1', 'b3gat1', 'seed')
('B3GAT1', 'CD57', 'cd57', 'seed')
('B3GAT1', 'HNK-1', 'hnk1', 'seed')
('EPCAM', 'Epithelial Cell Adhesion Molecule', 'epithelialcelladhesionmolecule', 'custom:demo')
('EPCAM', 'ep-cam', 'epcam', 'custom:demo')


## 6) Batch resolution workflow

This pattern is useful for processing antibody panels and preserving reproducible outputs.

In [8]:
markers = [
    'ki-67', 'Ki67', 'SMA', 'CD57',
    'a-sma', 'ep-cam', 'unknown_marker'
]

batch_results = []
for marker in markers:
    r = resolver.resolve(marker)
    batch_results.append(
        {
            'input_marker': marker,
            'normalized': r.normalized_query,
            'gene_symbol': r.gene_symbol,
            'matched_alias': r.matched_alias,
            'source': r.source,
            'is_found': r.gene_symbol is not None,
        }
    )

try:
    import pandas as pd
    df = pd.DataFrame(batch_results)
    df
except Exception:
    # Fallback when pandas is not installed.
    for row in batch_results:
        print(row)

## 7) Optional: bootstrap aliases from Ensembl

This section is optional and requires the extra dependency:

```bash
pip install "rpg_conv[ensembl]"
```

It is wrapped in `try/except` so the notebook remains runnable without extras.

In [9]:
try:
    from rpg_conv.ensembl_import import bootstrap_from_ensembl

    # Example: fetch and ingest human synonyms.
    # This can take time and requires internet access.
    imported_count = bootstrap_from_ensembl(resolver.connection, species='human')
    print(f'Imported {imported_count} alias rows from Ensembl (human).')
except Exception as e:
    print('Skipped Ensembl bootstrap (dependency/network not available).')
    print('Reason:', e)

Skipped Ensembl bootstrap (dependency/network not available).
Reason: 'external_gene_name'


## 8) Reproducible export of results

Store batch output as a CSV artifact for downstream analyses.

In [10]:
output_path = Path('./demo_batch_resolution_results.csv')

try:
    import pandas as pd
    pd.DataFrame(batch_results).to_csv(output_path, index=False)
except Exception:
    # No pandas: write a minimal CSV manually.
    import csv
    keys = list(batch_results[0].keys())
    with output_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(batch_results)

print(f'Wrote: {output_path.resolve()}')

Wrote: /Users/k23030440/robust_protein_gene/demo_batch_resolution_results.csv


## 9) Clean shutdown

Close the resolver connection explicitly at the end of notebook execution.

In [ ]:
resolver.close()
print('Resolver closed.')